# 02 — ChromaDB Basics

**Goal**: Learn how vector databases work by using ChromaDB.

ChromaDB is a free, open-source vector database written in Python.
It stores embeddings + metadata and lets you search by similarity.

In [ ]:
# !pip install chromadb

import chromadb
from chromadb.config import Settings
import requests
import numpy as np

print("Imports OK")

## 1. Creating a ChromaDB Collection

A collection = a table of embeddings with metadata.

In [ ]:
# Create client (persistent storage)
client = chromadb.PersistentClient(path="./chroma_data")

# Create or get a collection
collection = client.get_or_create_collection(name="genai_guide")

print(f"Collection '{collection.name}' ready")

## 2. Adding Documents

Each document needs:
- `id`: unique string
- `embedding`: vector (list of floats)
- `metadata`: dict (for filtering)
- `document`: original text (optional, stored for retrieval)

In [ ]:
def get_embedding(text):
    """Get embedding from Ollama."""
    resp = requests.post("http://localhost:11434/api/embeddings", json={
        "model": "nomic-embed-text",
        "prompt": text
    })
    return resp.json()["embedding"]

# Our documents
documents = [
    {"id": "doc1", "text": "Transformers use self-attention to process sequences", "topic": "architecture"},
    {"id": "doc2", "text": "RAG combines document retrieval with LLM generation", "topic": "rag"},
    {"id": "doc3", "text": "Fine-tuning adapts pretrained models to specific tasks", "topic": "training"},
    {"id": "doc4", "text": "Vector databases store embeddings for similarity search", "topic": "infrastructure"},
    {"id": "doc5", "text": "Prompt engineering designs effective instructions for LLMs", "topic": "practical"},
]

# Add to ChromaDB
for doc in documents:
    embedding = get_embedding(doc["text"])
    collection.add(
        ids=[doc["id"]],
        embeddings=[embedding],
        metadatas=[{"topic": doc["topic"]}],
        documents=[doc["text"]]
    )

print(f"Added {len(documents)} documents")

## 3. Querying (Semantic Search)

In [ ]:
def search(query, n_results=3):
    """Search ChromaDB for similar documents."""
    query_emb = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=n_results
    )
    
    print(f"Query: {query}\n")
    for i in range(len(results['ids'][0])):
        print(f"{i+1}. [Dist: {results['distances'][0][i]:.3f}] {results['documents'][0][i]}")
        print(f"   Topic: {results['metadatas'][0][i]['topic']}")

search("How do I make my model better at a specific task?")

In [ ]:
search("What technology powers modern AI search?")


## 4. Filtering with Metadata

You can filter by metadata **before** or **after** the search.

In [ ]:
# Add more docs for richer filtering
more_docs = [
    {"id": "doc6", "text": "Backpropagation computes gradients through network layers", "topic": "theory"},
    {"id": "doc7", "text": "LangChain provides tools for LLM application development", "topic": "framework"},
    {"id": "doc8", "text": "LlamaIndex specializes in data indexing for LLMs", "topic": "framework"},
]

for doc in more_docs:
    embedding = get_embedding(doc["text"])
    collection.add(ids=[doc["id"]], embeddings=[embedding],
                  metadatas=[{"topic": doc["topic"]}], documents=[doc["text"]])

# Query with filter
query_emb = get_embedding("How do I build an LLM application?")
results = collection.query(
    query_embeddings=[query_emb],
    n_results=3,
    where={"topic": "framework"}  # only framework docs
)

print("Filtered: only 'framework' topic")
for doc in results['documents'][0]:
    print(f"  - {doc}")

## 5. Updating & Deleting

In [ ]:
# Update a document
new_emb = get_embedding("Transformers use self-attention mechanisms")
collection.update(
    ids=["doc1"],
    embeddings=[new_emb],
    metadatas=[{"topic": "architecture", "updated": True}],
    documents=["Transformers use self-attention mechanisms"]
)
print("Updated doc1")

# Delete a document
collection.delete(ids=["doc5"])
print("Deleted doc5")

## 6. Collection Stats

In [ ]:
count = collection.count()
print(f"Documents in collection: {count}")

# Peek at first few
print("\nAll documents:")
all_docs = collection.get()
for i, doc in enumerate(all_docs['documents']):
    print(f"  {all_docs['ids'][i]}: {doc[:50]}...")

## Key Takeaways

1. **ChromaDB** = free, local vector database
2. **Collections** = tables of embeddings + metadata
3. **Semantic search** = embed query → find nearest vectors
4. **Metadata filtering** = build real-world search apps
5. ChromaDB is the easiest vector DB to start with (Python-native)

**Next**: Advanced vector search techniques →